In [6]:
!pip install faiss-cpu -q

In [7]:
# MILESTONE 3

import os
import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline

# 1. Data Path Setup
DATA_PATH = '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'
train = pd.read_csv(DATA_PATH)
print(f"Successfully loaded training set. Total records: {len(train)}")

# 2. Knowledge Base Construction & Vectorization (FAISS Indexing)
print("\n Constructing Vector Database Knowledge Base.")
kb = []
for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))

bi_encoder = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = bi_encoder.encode(kb, show_progress_bar=False)

# Building standard dense flat L2 FAISS index
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)
print(f" FAISS vector index successfully populated with {index.ntotal} entries.")

def retrieve_top_k(prompt, k=5):
    q_emb = bi_encoder.encode([prompt], show_progress_bar=False)
    distances, indices = index.search(q_emb, k)
    return distances[0], indices[0]

# Q1: Baseline Zero-Shot Classifier 

zs_classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=-1)

row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])]
ans_150 = str(row_150[row_150['answer']])

res_q1 = zs_classifier(prompt_150, candidate_labels=labels_150)
q1_score = res_q1['scores'][res_q1['labels'].index(ans_150)]
print(f"Q1 Target Probability Score: {round(q1_score, 3)}")

# Q2: Bi-Encoder Dense Retrieval Capabilities (Row 150)
_, retrieved_indices_150 = retrieve_top_k(prompt_150, k=10)
retrieved_list_150 = list(retrieved_indices_150)

rank_q2 = retrieved_list_150.index(150) + 1 if 150 in retrieved_list_150 else None
print(f"Q2 True Document Rank (FAISS Retrieval): {rank_q2}")


# Q3: Modern Cross-Encoder Reranking Architecture (Row 150)
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

docs_10 = [kb[i] for i in retrieved_indices_150]
pairs = [[prompt_150, doc] for doc in docs_10]
ce_scores = cross_encoder.predict(pairs)

# Rank candidates descending by predicted attention mapping relevance
sorted_indices = [retrieved_indices_150[i] for i in np.argsort(ce_scores)[::-1]]
rank_q3 = sorted_indices.index(150) + 1 if 150 in sorted_indices else None
print(f"Q3 True Document Rank (Post Reranking): {rank_q3}")


# Q4: Context Sequence Window Exploration (Row 42 Token Tracking)
bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

row_42 = train.iloc[42]
prompt_42 = str(row_42['prompt'])
_, retrieved_idx_42 = retrieve_top_k(prompt_42, k=5)
concatenated_docs_42 = " ".join([kb[i] for i in retrieved_idx_42])
rag_string_42 = f"Context: {concatenated_docs_42} Question: {prompt_42}"

tokens_42 = bert_tokenizer(rag_string_42, truncation=False)['input_ids']
print(f"Q4 Untruncated Sequence Token Generation: {len(tokens_42)}")


# Q5: Golden Ground-Truth Context Injected RAG Pipeline (Row 150)
true_doc_150 = kb[150]
rag_string_150 = f"Context: {true_doc_150} Question: {prompt_150}"

res_q5 = zs_classifier(rag_string_150, candidate_labels=labels_150)
q5_score = res_q5['scores'][res_q5['labels'].index(ans_150)]
print(f"Q5 Ground-Truth Augmented Probability: {round(q5_score, 3)}")


# Q6: System Vulnerability Testing via Adversarial Injection (Row 150)
adversarial_doc = kb[999]  # Mismatched / unaligned data slice
adversarial_string = f"Context: {adversarial_doc} Question: {prompt_150}"

res_q6 = zs_classifier(adversarial_string, candidate_labels=labels_150)
q6_score = res_q6['scores'][res_q6['labels'].index(ans_150)]
print(f"Q6 Corrupted Context Probability Shift: {round(q6_score, 3)}")


# Q7: Multi-Document Vector Alignment & Database Hit Rate Evaluation
hits = 0
for i in range(100):
    row = train.iloc[i]
    prompt_i = str(row['prompt'])
    correct_text_i = str(row[row['answer']])

    _, retrieved_idx_i = retrieve_top_k(prompt_i, k=5)
    retrieved_docs_i = [kb[j] for j in retrieved_idx_i]

    if any(correct_text_i in doc for doc in retrieved_docs_i):
        hits += 1

hit_rate = round((hits / 100) * 100, 1)
print(f"Q7 Knowledge Base Retrieval Hit Rate: {hit_rate}%")


# Q8: Multi-Stage RAG Pipeline Optimization over 20 Slices & MAP@3 Tracking
def calculate_ap_at_3(ranked_letters, correct_letter):
    for rank, letter in enumerate(ranked_letters[:3], start=1):
        if letter == correct_letter:
            return 1.0 / rank
    return 0.0

ap_scores = []
for i in range(20):
    row = train.iloc[i]
    prompt_i = str(row['prompt'])
    letters = ['A', 'B', 'C', 'D', 'E']
    option_texts = [str(row[l]) for l in letters]
    correct_letter_i = row['answer']
    # Retrieval Stage
    _, retrieved_idx_i = retrieve_top_k(prompt_i, k=5)
    docs_5 = [kb[j] for j in retrieved_idx_i]
    # Reranking Stage via Cross-Encoder Attention Mechanics
    pairs_i = [[prompt_i, doc] for doc in docs_5]
    scores_i = cross_encoder.predict(pairs_i)
    best_doc = docs_5[int(np.argmax(scores_i))]
    # Prompt Augmentation
    rag_str = f"Context: {best_doc} Question: {prompt_i}"
    # Downstream Classification Framework Inference
    result_i = zs_classifier(rag_str, candidate_labels=option_texts)
    # Scoring Vector Formulation & Alphabetical Map Sorting
    text_to_letter = dict(zip(option_texts, letters))
    ranked_letters = [text_to_letter[lbl] for lbl in result_i['labels']]
    ap = calculate_ap_at_3(ranked_letters, correct_letter_i)
    ap_scores.append(ap)

map_at_3 = round(float(np.mean(ap_scores)), 3)
print(f"Q8 Strategic RAG Engine Pipeline MAP@3 Score: {map_at_3}")


# SUMMARY
print(f"Q1 : {round(q1_score, 3)}")
print(f"Q2 : {rank_q2}")
print(f"Q3 : {rank_q3}")
print(f"Q4 : {q4_token_count if 'q4_token_count' in locals() else len(tokens_42)}")
print(f"Q5 : {round(q5_score, 3)}")
print(f"Q6 : {round(q6_score, 3)}")
print(f"Q7 : {hit_rate}")
print(f"Q8 : {map_at_3}")

Successfully loaded training set. Total records: 2000

 Constructing Vector Database Knowledge Base.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 FAISS vector index successfully populated with 2000 entries.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Q1 Target Probability Score: 0.384
Q2 True Document Rank (FAISS Retrieval): 10


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Q3 True Document Rank (Post Reranking): 1
Q4 Untruncated Sequence Token Generation: 216
Q5 Ground-Truth Augmented Probability: 0.989
Q6 Corrupted Context Probability Shift: 0.529
Q7 Knowledge Base Retrieval Hit Rate: 73.0%
Q8 Strategic RAG Engine Pipeline MAP@3 Score: 0.975
Q1 : 0.384
Q2 : 10
Q3 : 1
Q4 : 216
Q5 : 0.989
Q6 : 0.529
Q7 : 73.0
Q8 : 0.975
